# LeetCode #1129: Shortest Path with Alternating Colors

https://leetcode.com/problems/shortest-path-with-alternating-colors/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS all paths)** | $O(2^{n+e})$ | $O(n+e)$ |
| **Optimal: BFS with Color State ★** | $O(n + e)$ | $O(n + e)$ |

---

## Understanding the Methods

### Brute Force (DFS all paths)
Explore every path from node 0 using DFS, tracking color alternation. Without visited-state tracking, paths can loop; even with it the combinatorics of color+node states can blow up exponentially for dense graphs.

### Optimal: BFS with Color State ★
BFS from node 0, but the "state" is `(node, last_color_used)`. Mark states `(node, color)` as visited separately — the same node reached first by a red edge and then by a blue edge should both be explored because they enable different continuations. BFS guarantees the first time a state is reached is the shortest path to it.

**Constraints:**
* 1 <= n <= 100
* 0 <= redEdges.length, blueEdges.length <= 300
* redEdges[i].length == blueEdges[i].length == 2

## Solutions
### C#

In [ ]:
// BFS with color state: (node, lastColor) to explore alternating paths
public class Solution {
    public int[] ShortestAlternatingPaths(int n, int[][] redEdges, int[][] blueEdges) {
        // Build adjacency lists: adj[node] = list of (neighbor, color) pairs
        // Color: 0 = red, 1 = blue
        var adj = new List<(int neighbor, int color)>[n];
        for (int i = 0; i < n; i++) adj[i] = new();
        foreach (var e in redEdges) adj[e[0]].Add((e[1], 0));
        foreach (var e in blueEdges) adj[e[0]].Add((e[1], 1));

        int[] dist = new int[n];
        Array.Fill(dist, -1);
        dist[0] = 0;

        // State: (node, lastColor). Use bool[n][2] to track visited states
        bool[,] visited = new bool[n, 2];
        // BFS queue holds (node, lastColor, distance)
        var queue = new Queue<(int node, int color, int d)>();
        // Start from node 0 with either color (we can leave on red or blue)
        queue.Enqueue((0, 0, 0));
        queue.Enqueue((0, 1, 0));
        visited[0, 0] = true;
        visited[0, 1] = true;

        while (queue.Count > 0) {
            var (node, color, d) = queue.Dequeue();
            // Must alternate: next edge must be the opposite color
            int nextColor = 1 - color;
            foreach (var (nb, ec) in adj[node]) {
                if (ec != nextColor || visited[nb, ec]) continue;
                visited[nb, ec] = true;
                if (dist[nb] == -1) dist[nb] = d + 1;
                queue.Enqueue((nb, ec, d + 1));
            }
        }
        return dist;
    }
}

### Python

In [ ]:
# BFS with color state: (node, lastColor) to explore alternating paths
from typing import List
from collections import deque, defaultdict

class Solution:
    def shortestAlternatingPaths(self, n: int, redEdges: List[List[int]], blueEdges: List[List[int]]) -> List[int]:
        # Build adjacency lists with color labels: 0=red, 1=blue
        adj = defaultdict(list)
        for u, v in redEdges:
            adj[u].append((v, 0))
        for u, v in blueEdges:
            adj[u].append((v, 1))

        dist = [-1] * n
        dist[0] = 0

        # BFS: state = (node, last_color_used)
        visited = [[False, False] for _ in range(n)]
        queue = deque()
        # Can depart node 0 on either color
        for color in (0, 1):
            queue.append((0, color, 0))
            visited[0][color] = True

        while queue:
            node, color, d = queue.popleft()
            # Next edge must be the opposite color
            next_color = 1 - color
            for nb, ec in adj[node]:
                if ec != next_color or visited[nb][ec]:
                    continue
                visited[nb][ec] = True
                if dist[nb] == -1:
                    dist[nb] = d + 1
                queue.append((nb, ec, d + 1))

        return dist

### Go

In [ ]:
// BFS with color state: (node, lastColor) to explore alternating paths
package main

func shortestAlternatingPaths(n int, redEdges [][]int, blueEdges [][]int) []int {
    type Edge struct{ neighbor, color int }
    adj := make([][]Edge, n)
    for _, e := range redEdges {
        adj[e[0]] = append(adj[e[0]], Edge{e[1], 0})
    }
    for _, e := range blueEdges {
        adj[e[0]] = append(adj[e[0]], Edge{e[1], 1})
    }

    dist := make([]int, n)
    for i := range dist { dist[i] = -1 }
    dist[0] = 0

    // Visited indexed by [node][color]
    visited := make([][2]bool, n)
    type State struct{ node, color, d int }
    queue := []State{{0, 0, 0}, {0, 1, 0}}
    visited[0][0] = true
    visited[0][1] = true

    for len(queue) > 0 {
        s := queue[0]
        queue = queue[1:]
        nextColor := 1 - s.color
        for _, e := range adj[s.node] {
            if e.color != nextColor || visited[e.neighbor][e.color] {
                continue
            }
            visited[e.neighbor][e.color] = true
            if dist[e.neighbor] == -1 {
                dist[e.neighbor] = s.d + 1
            }
            queue = append(queue, State{e.neighbor, e.color, s.d + 1})
        }
    }
    return dist
}

### Rust

In [ ]:
// BFS with color state: (node, lastColor) to explore alternating paths
use std::collections::VecDeque;

impl Solution {
    pub fn shortest_alternating_paths(n: i32, red_edges: Vec<Vec<i32>>, blue_edges: Vec<Vec<i32>>) -> Vec<i32> {
        let n = n as usize;
        // Adjacency list: (neighbor, color); 0=red, 1=blue
        let mut adj: Vec<Vec<(usize, usize)>> = vec![vec![]; n];
        for e in &red_edges { adj[e[0] as usize].push((e[1] as usize, 0)); }
        for e in &blue_edges { adj[e[0] as usize].push((e[1] as usize, 1)); }

        let mut dist = vec![-1i32; n];
        dist[0] = 0;
        // visited[node][color]
        let mut visited = vec![[false; 2]; n];
        visited[0][0] = true;
        visited[0][1] = true;

        // Queue: (node, last_color, distance)
        let mut queue: VecDeque<(usize, usize, i32)> = VecDeque::new();
        queue.push_back((0, 0, 0));
        queue.push_back((0, 1, 0));

        while let Some((node, color, d)) = queue.pop_front() {
            // Must alternate color on the next edge
            let next_color = 1 - color;
            for &(nb, ec) in &adj[node] {
                if ec != next_color || visited[nb][ec] { continue; }
                visited[nb][ec] = true;
                if dist[nb] == -1 { dist[nb] = d + 1; }
                queue.push_back((nb, ec, d + 1));
            }
        }
        dist
    }
}

## Example Scenarios

**1. Common Case** — Simple two-color graph

**Input:** `n=3, redEdges=[[0,1],[1,2]], blueEdges=[]`
Only red edges exist. From node 0, reach node 1 via red (distance 1). From node 1, the next edge must be blue — none exists, so node 2 is unreachable via alternating path. Result: `[0, 1, -1]`.

**2. Slightly Complex** — Both colors interleaved

**Input:** `n=3, redEdges=[[0,1]], blueEdges=[[1,2]]`
Node 0→1 via red (dist=1), then 1→2 via blue (dist=2). Alternating is satisfied. Result: `[0, 1, 2]`.

**3. Edge Case: Time Factor** — Dense graph with many edges

**Input:** `n=100, redEdges` and `blueEdges` each with 300 edges forming a complete bipartite-like structure
BFS visits each `(node, color)` state at most once: $2n = 200$ states. Even with 600 total edges, the total work is $O(n + e) = O(700)$.

**4. Edge Case: Space Factor** — Disconnected nodes

**Input:** `n=5, redEdges=[[0,1]], blueEdges=[[0,2]]`
Nodes 3 and 4 have no incoming edges. BFS enqueues only reachable states; visited array is allocated for all $n$ nodes but most entries stay false. Dist for unreachable nodes stays $-1$.

**5. Almost-Impossible but Plausible** — Self-loop or multi-edge

**Input:** `n=3, redEdges=[[0,1],[0,1]], blueEdges=[[1,0],[1,2]]`
The duplicate red edge 0→1 is harmless: once `(1, red)` is visited the duplicate is skipped. The blue edge 1→0 creates a back-edge, but `(0, blue)` is marked visited in the initial setup, so no cycle is followed. Result: `[0, 1, 2]`.